In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

import albumentations as A

import numpy as np
import matplotlib.pyplot as plt

import os
import cv2

In [80]:
class CustomDataset(Dataset):
    def __init__(self, path, method, transform = None):
        if method == 'train': self.path = os.path.join(path, 'train')
        elif method == 'valid': self.path =  os.path.join(path, 'valid')
        elif method == 'test': self.path = os.path.join(path, 'test')
        self.trasform = transform

        self.images, self.labels = os.listdir(self.path + '/images'), os.listdir(self.path + '/labels')
        self.data = []
        for img_file, lbl_file in zip(self.images, self.labels):

            img_path = os.path.join(self.path, 'images', img_file)
            lbl_path = os.path.join(self.path, 'labels', lbl_file)

            bboxes = []
            labels = []

            with open(lbl_path) as f:
                for line in f:
                    cls, x, y, w, h = map(float, line.strip().split())
                    bboxes.append([x,y,w,h])
                    labels.append(int(cls))

            
            self.data.append({
                'img_path': img_path,
                'bboxes': bboxes,
                'labels': labels
            })

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]

        img = cv2.imread(item['img_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        box = item['bboxes']
        labels = item['labels']

        return img, box, labels

In [ ]:
test_dataset = CustomDataset('data','test')
val_dataset = CustomDataset('data','valid')
train_dataset = CustomDataset('data','train')

In [83]:
train_data = DataLoader(test_dataset, batch_size=32, shuffle=True)
val_data = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_data = DataLoader(test_dataset, batch_size=32, shuffle=False)